# 04 - Trend analysis (Phase 4)

Pixel-wise **Mann-Kendall** and **Sen's slope** on the ANNUAL composite series,
Benjamini-Hochberg FDR correction, a decadal comparison, and
autocorrelation-robust trend tests on aggregate and per-division series.

## Why this notebook is the one that measures warming

Phase 3's UTFVI epoch maps **cannot** show warming. UTFVI's reference is the
year's own mean, so a uniformly warming city produces no class change at all;
epoch-to-epoch drift is *redistribution*. **Sen's slope here is the warming**,
and nothing in this notebook "confirms" the UTFVI maps.

## Two corrections to the GEE community tutorial

This project follows the official *Non-Parametric Trend Analysis* tutorial for
the MK variance, Z and p - with two deliberate corrections, both implemented in
`src/colombo_uhi/trends.py`:

| # | Tutorial | Why it is wrong here | What we do |
|---|---|---|---|
| 1 | `sign = diff.clamp(-1,1).int()` | `.int()` truncates toward zero, so a **+0.3 degC** year-to-year difference becomes sign **0**. Most annual LST differences in Colombo are well under 1 degC, so this collapses most of S. The tutorial states it is for *"discrete data (i.e. not floating point)"*. | `diff.gt(0).subtract(diff.lt(0))` - exact for any float |
| 2 | `p = 1 - Phi(abs(z))`, thresholded at 0.025 | That is the **one-sided** p. Benjamini-Hochberg needs **two-sided** input; the one-sided form halves every p and roughly doubles the significant area. | `mk_p_two_sided = 2*(1 - Phi(abs(Z)))` |

A third: the tutorial's tie correction detects ties by **exact float equality**.
On continuous LST there are effectively zero ties, so the term is measurably
zero while the `arraySort` machinery is expensive. `trends.tie_correction` is
off, with the reasoning recorded in `config/params.yaml`.

## Scope

* Pixel-wise trend rasters for **Landsat dry-season (Jan-Mar)** and **MODIS
  night (Terra + Aqua)**. MODIS *daytime* is excluded: Terra's orbital drift
  after ~2020 contaminates end-of-series daytime trends (Phase 2 finding).
* FDR twice: **in-session preview** (coarse, immediate) and on the **exported
  raster** (authoritative).
* Modified Mann-Kendall (Hamed & Rao) on the SUHII series and per-GN series.

## Non-negotiable caveats

1. This is **land surface temperature**, never air temperature.
2. Every trend product ships its per-pixel valid-**year** count (`n_years`).
3. `n_years` is never masked - an excluded pixel reads as *excluded*, not *missing*.
4. Landsat is a single ~10:30 overpass. Night trends come only from MODIS.

## If Earth Engine says "User memory limit exceeded"

In this order, and only the last one changes the numbers:

1. Lower `trends.batch_years` / `trends.zonal_batch_years` to 1.
2. Narrow the region (`trends.region: "cmc"`).
3. Raise `trends.fit_scale_m` (100 -> 300).
4. As a last resort, export the annual stack to an asset and set
   `trends.annual_stack_asset`.

**Never** set `trends.mk_method: "pairwise"` over the full AOI: a 26-year series
makes 325 pairwise images, each carrying two full composite graphs, and Colab
run 2 died on 26.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00, 01, 02 or 03 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "valid_obs_required", "single_overpass", "fdr_dependence"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 0 - prove the loaded code is Phase 4

Two checks, because the usual one is **not enough here**.

The name-based guard catches a stale `src/`. But Phase 4's change to
`composites.annual_composites` is **additive** - it sets a new `series_basis`
property and renames nothing - so a name check passes against a Phase-2
`composites.py` and then `require_annual_series()` fails much later with a
confusing message. The second cell therefore proves the **setter** is present.

In [ ]:
# COLAB: RUN THIS CELL
import ee

from colombo_uhi import aoi, composites, exports, landcover, landsat, modis, trends, uhi_metrics, viz

# Fail fast and legibly if a STALE colombo_uhi is loaded (see the purge above).
#
# THE RULE, learned the hard way in run 8: every entry must name a function
# introduced by the MOST RECENT revision of that module. Listing only functions
# that also existed in the previous revision makes this guard VACUOUS -- it
# passes, and you get old figures from new notebook cells with no error anywhere.
_required = {
    "aoi": ["rural_reference", "static_water_mask", "lcz_scope_geometry"],
    "composites": ["annual_composites", "warn_if_counts_are_empty"],
    "uhi_metrics": ["suhii_all_sources", "resolve_source", "source_collection"],
    # Everything below arrived with Phase 4. trends.py and exports.py were
    # DOCSTRING-ONLY STUBS before it, and landcover.py did not exist at all.
    "trends": [
        "annual_series", "require_annual_series", "validate_series_metadata",
        "fit_stack", "trend_image", "verify_trend_bands", "signum_array",
        "sens_slope_array", "mk_statistics_from_tau", "two_sided_p",
        "benjamini_hochberg", "fdr_significant_fraction", "mk_comparison",
        "suhii_trends", "decadal_means", "decadal_difference", "trend_by_class",
        "zonal_annual_series", "zonal_trend_table", "sample_trend_arrays",
        "read_trend_raster", "apply_fdr_to_raster",
    ],
    "exports": [
        "export_name", "resolve_export_settings", "image_to_drive",
        "table_to_drive", "describe_tasks", "wait_for_tasks", "find_tasks",
    ],
    "landcover": ["worldcover", "lcz_class_image", "stratified_stats", "class_labels"],
    "viz": ["trend_vis_params", "build_trend_map_figure", "build_mk_comparison_figure",
            "build_trend_by_class_figure"],
}
_absent = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_absent = {k: v for k, v in _absent.items() if v}
if _absent:
    raise RuntimeError(
        f"colombo_uhi modules are STALE, missing {_absent}.\n"
        f"  trends loaded from: {trends.__file__}\n"
        f"  exports loaded from: {exports.__file__}\n"
        "Fix, in order:\n"
        "  1. MOST LIKELY: local changes are COMMITTED BUT NOT PUSHED, or not\n"
        "     committed at all. This notebook runs against the pushed repo, so\n"
        "     editing a file locally is not enough. Check that the HEAD line\n"
        "     printed by the clone cell is the revision you expect, then\n"
        "     git push and re-run from the CLONE cell.\n"
        "  2. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  3. If you uploaded this .ipynb by hand rather than opening it from\n"
        "     the repo, the notebook and src/ can be at DIFFERENT revisions -\n"
        "     that combination produces stale figures with no error at all."
    )
print("PASS: all Phase 4 functions are present.")

TREND_SOURCE = params["trends"]["pixel_sources"][0]
FIT_SCALE = params["trends"]["fit_scale_m"]
PIXEL_SOURCES = list(params["trends"]["pixel_sources"])
print("Headline source:", TREND_SOURCE, "| fit scale:", FIT_SCALE, "m")
print("Pixel-wise sources:", PIXEL_SOURCES)

In [ ]:
# COLAB: RUN THIS CELL
# The name guard above CANNOT see an additive change. Phase 4 added a
# `series_basis` property to composites.annual_composites without renaming
# anything, so prove the SETTER is there. One cheap round trip on one year.
_probe_region = aoi.cmc_boundary(params)
_probe_scenes = uhi_metrics.source_collection(TREND_SOURCE, params, region=_probe_region)
_probe = composites.annual_composites(
    _probe_scenes, params, with_percentile=False, start_year=2024, end_year=2024
).first()

_basis_prop = params["composites"]["series_basis_property"]
_basis = _probe.get(_basis_prop).getInfo()
_window = _probe.get(params["composites"]["window_months_property"]).getInfo()

print(f"{_basis_prop} = {_basis!r}   window_months = {_window!r}")
if _basis != params["trends"]["series_basis"]:
    raise RuntimeError(
        f"composites.annual_composites did not set {_basis_prop} (got {_basis!r}). "
        "Your src/ predates Phase 4 - re-run the CLONE cell, then the module-purge "
        "cell, then this one."
    )
print("PASS: the Phase 4 series_basis marker is being set.")

## Step 1 - geometries, the cheap water mask, and the work region

| Decision | Why |
|---|---|
| `STATIC_WATER = aoi.static_water_mask(...)` | `aoi.water_mask` composites ~100 Landsat scenes internally and is re-instantiated for **every** image it masks. Over a 26-year series that is unaffordable. The measured cost of the substitution is **-0.074 degC** on the CMC 2025 mean - quote it, do not re-derive it. |
| `WORK_REGION` = Colombo District | `aoi.analysis_region` (Western Province + 25 km) is ~20x the pixels for nothing. |
| `FIT_SCALE = 100 m` | Adjacent 30 m LST pixels are near-duplicates, so fitting at 30 m multiplies the FDR test count ~11x without adding independent information - and drags the BH threshold down for every genuine pixel. |

In [ ]:
# COLAB: RUN THIS CELL
district_fc = aoi.colombo_district(params)
WORK_REGION = district_fc.geometry()
CMC = aoi.cmc_boundary(params)
STATIC_WATER = aoi.static_water_mask(params, region=WORK_REGION)

print("District area (km2):", round(aoi.area_km2(WORK_REGION).getInfo(), 2))
print("CMC area (km2)     :", round(aoi.area_km2(CMC).getInfo(), 2))
print()
print("REMINDER: any CMC area must be quoted WITH its reduction scale - the")
print("land-only area is 40.18 km2 at 30 m and 37.70 km2 at 300 m.")

## Step 2 - PROBE the reducers before anything else runs

Three unknowns are settled here, on a **tiny box**, before a single full-AOI
computation. Each costs seconds; getting one wrong costs a 26-year run.

1. **Band names.** `ee.Reducer.sensSlope` and `ee.Reducer.kendallsCorrelation`
   name their outputs differently depending on whether the reducer reports one
   input or several (`composites._composite_reducer` documents the same rule for
   `sharedInputs`). We use `numInputs=2` so both should emit **bare** names.
2. **Sen's input order.** `sensSlope` takes **x then y**. Reversed, it returns
   the **reciprocal** slope - not an error. A collection of known slope 2.0
   settles it: if the probe returns 0.5, swap `trends.sen_input_order`.
3. **Is the reducer's own p-value one- or two-sided?** Compared against `scipy`
   on the same numbers. Our `mk_p_two_sided` is derived from Z and is two-sided
   regardless; this only tells us how to read `mk_p_ee`.

**Record the output of these cells in `PROGRESS.md`** so nobody has to repeat it.

In [ ]:
# COLAB: RUN THIS CELL
# 2a - band names, over a ~2 km box so nothing evaluates the full AOI.
_box = CMC.centroid(maxError=1).buffer(1000).bounds()
_probe_series = trends.annual_series(
    TREND_SOURCE, params, region=_box, start_year=2020, end_year=2025
)
_probe_stack = trends.fit_stack(_probe_series, params, start_year=2020, validate=False)

print("fit stack bands        :", _probe_stack.first().bandNames().getInfo())
print("sensSlope outputs      :",
      _probe_stack.reduce(ee.Reducer.sensSlope()).bandNames().getInfo())
print("kendallsCorrelation(2) :",
      _probe_stack.reduce(ee.Reducer.kendallsCorrelation(2)).bandNames().getInfo())
print("kendallsCorrelation(1) :",
      _probe_stack.select([trends.FIT_Y_BAND])
      .reduce(ee.Reducer.kendallsCorrelation(1)).bandNames().getInfo())
print()
print("COPY THE ABOVE INTO PROGRESS.md. If they are not what trends.bands and")
print("trends.mk_num_inputs assume, edit config/params.yaml, commit, push, and")
print("re-run from the CLONE cell.")

In [ ]:
# COLAB: RUN THIS CELL
# 2b - Sen's input ORDER, against a collection whose slope is exactly 2.0.
# A reversed order returns the RECIPROCAL (0.5), not an error.
_known = ee.ImageCollection([
    ee.Image.cat([
        ee.Image.constant(x).toDouble().rename(trends.FIT_X_BAND),
        ee.Image.constant(10.0 + 2.0 * x).toDouble().rename(trends.FIT_Y_BAND),
    ])
    for x in (0, 1, 2, 3, 4)
])
_sen = _known.reduce(ee.Reducer.sensSlope())
_result = _sen.reduceRegion(
    reducer=ee.Reducer.first(), geometry=_box, scale=1000, maxPixels=1e9
).getInfo()
print("known slope 2.0 ->", _result)

_slope_value = next((v for k, v in _result.items() if "slope" in k.lower()), None)
if _slope_value is not None and abs(_slope_value - 2.0) < 1e-6:
    print("PASS: sensSlope takes x then y, as trends.sen_input_order assumes.")
elif _slope_value is not None and abs(_slope_value - 0.5) < 1e-6:
    raise RuntimeError(
        "FAIL: sensSlope returned the RECIPROCAL slope (0.5 for a true 2.0), so "
        "its inputs are y then x. Swap trends.sen_input_order in "
        "config/params.yaml to ['y', 'x'], commit, push, re-run from the CLONE cell."
    )
else:
    raise RuntimeError(f"FAIL: unexpected sensSlope output {_result}")

In [ ]:
# COLAB: RUN THIS CELL
# 2c - does kendallsCorrelation agree with scipy/pymannkendall on tau, and is
# its own p-value one- or two-sided?
#
# Run through the SAME code path the pipeline uses: an ImageCollection of
# two-band [x, y] images. Do NOT use ee.List.reduce here - a multi-input reducer
# over an ee.List wants a list of PAIRS [[x0,y0],[x1,y1],...], and passing two
# parallel arrays instead silently reduces 2 "samples" and returns tau = 1 with
# a NaN p-value.
import numpy as np
import pymannkendall as pmk
from scipy import stats

_values = [26.1, 26.4, 26.3, 26.9, 27.0, 27.4, 27.2, 27.9, 28.1, 28.4, 28.3, 28.9]
_x = list(range(len(_values)))

_toy = ee.ImageCollection([
    ee.Image.cat([
        ee.Image.constant(float(_xi)).toDouble().rename(trends.FIT_X_BAND),
        ee.Image.constant(float(_yi)).toDouble().rename(trends.FIT_Y_BAND),
    ])
    for _xi, _yi in zip(_x, _values)
])
_reduced = _toy.reduce(ee.Reducer.kendallsCorrelation(2))
print("kendall output bands:", _reduced.bandNames().getInfo())

_ee_stats = _reduced.reduceRegion(
    reducer=ee.Reducer.first(), geometry=_box, scale=1000, maxPixels=1e9
).getInfo()
print("ee raw                    :", _ee_stats)


# Earth Engine returns NaN through JSON as the STRING 'NaN', not a float, so a
# bare `if value:` is truthy and the next arithmetic raises a TypeError.
def _as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


_ee_tau = _as_float(next((v for k, v in _ee_stats.items() if "tau" in k.lower()), None))
_ee_p = _as_float(
    next((v for k, v in _ee_stats.items() if "p" in k.lower() and "tau" not in k.lower()), None)
)

_sci = stats.kendalltau(_x, _values)
_pmk = pmk.original_test(_values)
_ours = trends.mk_statistics_from_tau(float(_sci.statistic), len(_values))

print()
print(f"  tau   scipy {float(_sci.statistic):.6f} | pmk {float(_pmk.Tau):.6f} | ee {_ee_tau:.6f}")
if np.isfinite(_ee_tau) and abs(_ee_tau - float(_sci.statistic)) < 1e-4:
    print("  PASS: ee tau matches scipy.")
else:
    raise RuntimeError(
        f"FAIL: ee tau {_ee_tau} does not match scipy {float(_sci.statistic):.6f}. "
        "If ee tau is exactly 1 or -1 the inputs were not paired correctly. Do "
        "not trust any trend product until this agrees."
    )

In [ ]:
# COLAB: RUN THIS CELL
# 2c (continued) - the p-value comparison.
#
# NOTE the two local p-values differ LEGITIMATELY: scipy's kendalltau uses an
# EXACT method at n=12, while pymannkendall - and our trends.two_sided_p - use
# the normal approximation with a continuity correction. Ours must match
# pymannkendall, NOT scipy's exact p. The exact value is printed only as a
# reminder that the difference is expected and is not a bug.
_normal_two_sided = float(_pmk.p)

print(f"  p  TWO-sided, normal approx (pmk) : {_normal_two_sided:.8f}")
print(f"  p  TWO-sided, ours from Z         : {float(_ours['p']):.8f}")
print(f"  p  ONE-sided would be             : {_normal_two_sided / 2:.8f}")
print(f"  p  scipy EXACT (different method) : {float(_sci.pvalue):.8f}")
print(f"  p  ee reducer                     : {_ee_p}")
print()

if abs(float(_ours["p"]) - _normal_two_sided) < 1e-6:
    print("  PASS: our two-sided p matches pymannkendall's normal approximation.")
else:
    raise RuntimeError(
        f"FAIL: our p {float(_ours['p'])} != pymannkendall {_normal_two_sided}."
    )

print()
if not np.isfinite(_ee_p):
    print("  ee p-value is NaN on this series.")
    print("  NOT A PROBLEM: mk_p_ee is exported for COMPARISON ONLY and is never")
    print("  fed to the FDR correction. Our mk_p_two_sided is derived from Z and")
    print("  is unaffected. Record 'ee p unusable' in PROGRESS.md and move on.")
else:
    _ratio = _normal_two_sided / _ee_p
    print(f"  ratio two_sided / ee_p = {_ratio:.3f}")
    if abs(_ratio - 2.0) < 0.3:
        print("  MEASURED: the ee reducer's p is ONE-sided.")
    elif abs(_ratio - 1.0) < 0.3:
        print("  MEASURED: the ee reducer's p is TWO-sided.")
    else:
        print("  MEASURED: the ee reducer's p matches neither cleanly - it likely")
        print("  uses a different tie/exact convention. Treat mk_p_ee as opaque.")
print()
print("Either way our exported mk_p_two_sided is ALWAYS two-sided, because it is")
print("derived from Z. Record the verdict in PROGRESS.md so nobody repeats this.")

## Step 3 - the structural guard, demonstrated

The brief requires that running Mann-Kendall on the raw sub-annual stack be
**structurally impossible**, not merely discouraged. Two layers:

* **Layer 1** - `trends.trend_image()` takes a *source key*, not a collection,
  and builds its own annual series. There is no parameter through which a scene
  stack can reach the reducers.
* **Layer 2** - `trends.require_annual_series()` refuses anything that is not
  one image per year, in ascending order, carrying the `series_basis` marker.

The cell below proves layer 2 *rejects* a scene collection and *accepts* the
dry-season annual series.

**The guard SAMPLES.** Reading a collection's properties forces Earth Engine to
materialise the images they hang off, and an annual series carries a full
composite graph per year. Inspecting all 26 in one request exceeds the memory
limit — and Colab run 10 measured that this happens **regardless of region**, so
the cost is the graph, not the pixels. A 2 km box does not rescue it.

Sampling is sound rather than a shortcut, because `annual_composites` emits one
image per calendar year *including empty ones*. So:

* **any** gap or extra image changes `size`, which is cheap to read;
* a scene stack fails on `size` (1674 against 26) **and** on the first image
  inspected (scenes carry `month`, composites carry `series_basis`).

Pass `full=True` to inspect every year in batches, at one round trip per batch.

`trend_image()` passes `validate=False` for a different reason: it takes a source
*key* and builds the series itself, so layer 1 has already guaranteed provenance
and layer 2 would only re-establish what is already certain.

In [ ]:
# COLAB: RUN THIS CELL
# The scene collection MUST be rejected.
_scenes = uhi_metrics.source_collection(TREND_SOURCE, params, region=_box)
try:
    trends.require_annual_series(_scenes.limit(40), params)
except ValueError as error:
    print("PASS: scene collection rejected.")
    print("  ->", str(error).split(".")[0])
else:
    raise RuntimeError(
        "FAIL: require_annual_series ACCEPTED a scene collection. The structural "
        "guard is broken - do not run any trend product until it is fixed."
    )

print()
# The annual series MUST be accepted. Sampled, not exhaustive - see the note
# above for why that is sound and why exhaustive is unaffordable.
SERIES = trends.annual_series(TREND_SOURCE, params, region=WORK_REGION)
_summary = trends.require_annual_series(
    SERIES, params,
    start_year=params["time"]["start_year"],
    end_year=params["time"]["end_year"],
)
print("PASS: annual series accepted.")
print("  ", _summary)
print()
print(f"Inspected {_summary['n_probed']} of {_summary['n_years']} images. The")
print("size check covers the rest: one image per calendar year is guaranteed by")
print("construction, so any gap would change n_years.")

## Step 4 - the Sen's slope map

`trends.trend_image()` returns every band in `trends.export_band_order`. The
statistical bands are masked where `n_years < trends.min_years`; **`n_years`
itself is never masked**, so an excluded pixel reads as *excluded* rather than
*missing* (caveat 2).

`verify_trend_bands` then does what a name check cannot: **tau is negative about
half the time and a p-value never is**, so an output *order* swap that correct
band names would hide shows up as a range violation.

In [ ]:
# COLAB: RUN THIS CELL
import time

_t0 = time.time()
TREND = trends.trend_image(TREND_SOURCE, params, region=WORK_REGION).clip(WORK_REGION)
print("bands:", TREND.bandNames().getInfo())

_ranges = trends.verify_trend_bands(TREND, params, _box, scale_m=FIT_SCALE)
print()
for _name, (_lo, _hi) in _ranges.items():
    print(f"  {_name:18s} min={_lo}  max={_hi}")
print()
print("PASS: every band is inside its mathematically possible range.")
print(f"({time.time() - _t0:.0f} s)")

In [ ]:
# COLAB: RUN THIS CELL
# Slope percentiles over the CMC, to check the palette stretch is not saturating.
_bands = params["trends"]["bands"]
_pcts = TREND.select([_bands["sen_slope"]]).reduceRegion(
    reducer=ee.Reducer.percentile([1, 5, 25, 50, 75, 95, 99]),
    geometry=CMC,
    scale=FIT_SCALE,
    maxPixels=params["composites"]["reduce_max_pixels"],
    tileScale=params["composites"]["tile_scale"],
).getInfo()
print("Sen's slope percentiles over the CMC (degC/yr):")
for _k, _v in sorted(_pcts.items()):
    print(f"  {_k:24s} {_v}")

_vis = viz.trend_vis_params(params)
print()
print(f"Palette stretch: {_vis['min']} .. {_vis['max']} degC/yr")
print("If more than ~5% of pixels sit outside that range, widen trends.slope_vis")
print("- a saturated map understates the extremes it exists to show.")

In [ ]:
# COLAB: RUN THIS CELL
from IPython.display import Image, display

os.makedirs("figures", exist_ok=True)
_outline = viz.outline_image(CMC, "000000", 2)

_slope_png = viz.save_thumbnail(
    [TREND.select([_bands["sen_slope"]]).visualize(**viz.trend_vis_params(params)), _outline],
    WORK_REGION, f"figures/trend_sen_slope_{TREND_SOURCE}_2000_2025.png",
)
print("Wrote", _slope_png)
display(Image(filename=str(_slope_png)))

_n_png = viz.save_thumbnail(
    [TREND.select([_bands["n_years"]]).visualize(**params["trends"]["n_years_vis"]), _outline],
    WORK_REGION, f"figures/trend_n_years_{TREND_SOURCE}_2000_2025.png",
)
print("Wrote", _n_png)
print("CAVEAT 2: read the slope map against THIS one. Phase 2 measured WRS-2")
print("side-lap banding in obs_count - one overlap strip runs through the CMC -")
print("so trend confidence varies with orbit geometry as well as cloud, and the")
print("significance map will inherit those stripes.")
display(Image(filename=str(_n_png)))

## Step 5 - export the trend rasters to Drive

CLAUDE.md requires the FDR correction to be applied **in Python, on the exported
p-value raster**. That is the authoritative product. Step 6 computes an
in-session preview so the map can be seen now; Step 10 reads the export back.

`exports.image_to_drive` selects the image into `trends.export_band_order`
before exporting - band identity in a GeoTIFF is **positional**, so this is what
keeps the writer and the reader describing the same file.

In [ ]:
# COLAB: RUN THIS CELL
TREND_IMAGES = {TREND_SOURCE: TREND}
TASKS = []

for _key in PIXEL_SOURCES:
    if _key not in TREND_IMAGES:
        TREND_IMAGES[_key] = trends.trend_image(
            _key, params, region=WORK_REGION
        ).clip(WORK_REGION)
    _task = exports.image_to_drive(
        TREND_IMAGES[_key],
        product="lst_trend",
        aoi="district",
        params=params,
        region=WORK_REGION.bounds(),
        band_order=params["trends"]["export_band_order"],
        scale_m=FIT_SCALE,
        suffix=_key,
    )
    TASKS.append(_task)
    print("submitted:", _task.status().get("description"))

print()
print(f"{len(TASKS)} export task(s) queued to Drive folder "
      f"'{params['exports']['drive_folder']}'.")
print("Re-run the NEXT cell until every state reads COMPLETED. Meanwhile Step 6")
print("computes the in-session preview, so you do not have to wait here.")

In [ ]:
# COLAB: RUN THIS CELL  (re-runnable - poll until every state reads COMPLETED)
exports.describe_tasks(TASKS)

## Step 6 - FDR, previewed in-session

Benjamini-Hochberg cannot be done server-side: it needs every p-value at once,
sorted. This step pulls the trend bands into numpy at
`trends.fdr.preview_scale_m` (deliberately coarser than the fit scale - the
preview is for *shape*), applies BH, and reports the significant area.

**Two denominators are reported and both matter.** Untested pixels - cloud
starved, water masked, or below the minimum-year floor - are neither significant
nor non-significant, so quoting `fraction_of_tested` alone overstates coverage
and `fraction_of_total` alone understates the trend.

**Benjamini-Yekutieli is reported beside BH.** BH controls the FDR under
independence or positive regression dependency; a 100 m LST raster is strongly
spatially autocorrelated, so the *realised* false-discovery proportion varies far
more than its controlled expectation. BY is valid under arbitrary dependence and
is the honest upper bound. The pair is the sensitivity, exactly as Phase 3
reports the two rural definitions.

In [ ]:
# COLAB: RUN THIS CELL
import pandas as pd

_preview_scale = params["trends"]["fdr"]["preview_scale_m"]
ARRAYS = trends.sample_trend_arrays(
    TREND, params, region=CMC.bounds(), scale_m=_preview_scale
)
for _name, _array in ARRAYS.items():
    print(f"  {_name:18s} shape={_array.shape}  finite={int(np.isfinite(_array).sum()):,}")

_slope = ARRAYS[_bands["sen_slope"]]
_p = ARRAYS[_bands["mk_p_two_sided"]]
_n = ARRAYS[_bands["n_years"]]

TESTED = trends.trend_validity_mask(_p, _slope, _n, params)
MASKED_P = np.where(TESTED, _p, np.nan)

_rows = []
for _method in params["trends"]["fdr"]["sensitivity_methods"]:
    _rows.append(
        trends.fdr_significant_fraction(
            MASKED_P, params, method=_method, slope=_slope,
            pixel_area_m2=_preview_scale ** 2,
        )
    )
FDR_PREVIEW = pd.DataFrame(_rows)
FDR_PREVIEW

In [ ]:
# COLAB: RUN THIS CELL
_headline = FDR_PREVIEW[
    FDR_PREVIEW["method"] == params["trends"]["fdr"]["method"]
].iloc[0]
_conservative = FDR_PREVIEW[
    FDR_PREVIEW["method"] == "benjamini_yekutieli"
].iloc[0]

print("IN-SESSION PREVIEW over the CMC at", _preview_scale, "m")
print(f"  pixels total          : {_headline['n_total']:,}")
print(f"  pixels TESTED         : {_headline['n_tested']:,}")
print(f"  significant (BH)      : {_headline['n_significant']:,} "
      f"({_headline['fraction_of_tested']:.1%} of tested, "
      f"{_headline['fraction_of_total']:.1%} of total)")
print(f"  significant (BY)      : {_conservative['n_significant']:,} "
      f"({_conservative['fraction_of_tested']:.1%} of tested)")
print(f"  warming / cooling     : {_headline['n_warming']:,} / {_headline['n_cooling']:,}")
print()
print("Report the BH and BY figures TOGETHER. Neither is 'the' answer: BH is the")
print("headline, BY is the honest upper bound under arbitrary spatial dependence.")
print()
print("This is the PREVIEW at a coarse scale. The authoritative number comes from")
print("Step 10, on the exported raster at", FIT_SCALE, "m.")

In [ ]:
# COLAB: RUN THIS CELL
_reject, _adjusted = trends.benjamini_hochberg(MASKED_P, params=params)
_fig = viz.plot_trend_map(
    {
        "sen_slope": _slope,
        "significant": np.where(TESTED, _reject.astype("float64"), np.nan),
    },
    f"figures/trend_fdr_preview_{TREND_SOURCE}_2000_2025.png",
    params,
    title=f"Sen's slope, {TREND_SOURCE}, 2000-2025 (in-session preview, {_preview_scale} m)",
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

## Step 7 - MODIS night-time trends

Landsat sees a single ~10:30 overpass, so **night-time UHI can only come from
MODIS** (caveat 4). Two caveats travel with these maps:

* MODIS is a **coarse-unit** statistic - roughly 40 MODIS pixels cover the whole
  CMC, and they are edge-contaminated.
* Day and night are **not an equal-confidence pair**: night LST carries up to 3 K
  accepted uncertainty against 1 K for day.

MODIS *daytime* trends are deliberately absent from `trends.pixel_sources`:
Terra's orbital drift after ~2020 moves the overpass time and contaminates
end-of-series daytime trends. Night is unaffected by that drift.

In [ ]:
# COLAB: RUN THIS CELL
for _key in [k for k in PIXEL_SOURCES if k != TREND_SOURCE]:
    _img = TREND_IMAGES[_key]
    _png = viz.save_thumbnail(
        [_img.select([_bands["sen_slope"]]).visualize(**viz.trend_vis_params(params)), _outline],
        WORK_REGION, f"figures/trend_sen_slope_{_key}_2000_2025.png",
    )
    _mean = _img.select([_bands["sen_slope"]]).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=CMC, scale=1000,
        maxPixels=params["composites"]["reduce_max_pixels"],
        tileScale=params["composites"]["tile_scale"],
    ).getInfo()
    print(f"{_key:14s} mean CMC slope = {_mean.get(_bands['sen_slope'])} degC/yr")
    print("  wrote", _png)
    display(Image(filename=str(_png)))

## Step 8 - decadal means and difference maps

**The windows are 11 / 10 / 5 years - unequal by construction**, because the
study period ends in 2025. The 2021-2025 mean rests on roughly half the sample
of the others, so its standard error is about sqrt(2) larger and it dominates the
uncertainty of any difference map it appears in. That is why `decadal_means`
emits `sd_` and `n_years_` per window and `decadal_difference` emits `diff_se`
and `diff_z` - the asymmetry lands on the map, not in a footnote.

**The headline warming number is the Sen's slope, not a decadal difference.** A
decadal difference conflates trend with interannual variability (one hot ENSO
year at either end moves it) and with changing observation counts. These maps
exist to show *where* the warming is concentrated.

Note also that the decadal mean averages the **annual composites**, so every year
counts once. That is deliberately the opposite of `uhi_metrics.epoch_composite`,
which composites all the window's scenes and so weights years by scene count -
correct for UTFVI, wrong here, because it would make part of any decade-to-decade
difference a change in the Landsat constellation rather than in temperature.

In [ ]:
# COLAB: RUN THIS CELL
DECADES = trends.resolve_decades(None, params)
print("Decadal windows:")
for _label, _start, _end in DECADES:
    print(f"  {_label:12s} {_start}-{_end}  ({_end - _start + 1} years)")
print()
print("UNEQUAL BY DESIGN - the study period ends in 2025.")

MEANS = trends.decadal_means(TREND_SOURCE, params, region=WORK_REGION).clip(WORK_REGION)
print()
print("bands:", MEANS.bandNames().getInfo())

In [ ]:
# COLAB: RUN THIS CELL
_labels = [label for label, _, _ in DECADES]
DIFFS = {}
for _later, _earlier in ((_labels[1], _labels[0]), (_labels[2], _labels[1])):
    DIFFS[f"{_later}_minus_{_earlier}"] = trends.decadal_difference(
        MEANS, params, later=_later, earlier=_earlier
    )

_diff_vis = {"min": -2.0, "max": 2.0, "palette": params["trends"]["slope_vis"]["palette"]}
for _name, _image in DIFFS.items():
    _png = viz.save_thumbnail(
        [_image.select([0]).visualize(**_diff_vis), _outline],
        WORK_REGION, f"figures/trend_decadal_{_name}.png",
    )
    _stats = _image.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=CMC, scale=FIT_SCALE,
        maxPixels=params["composites"]["reduce_max_pixels"],
        tileScale=params["composites"]["tile_scale"],
    ).getInfo()
    print(_name, "-> CMC mean:", {k: (round(v, 3) if isinstance(v, float) else v)
                                  for k, v in _stats.items()})
    display(Image(filename=str(_png)))

## Step 9 - modified Mann-Kendall on the SUHII and per-GN series

Annual LST is positively autocorrelated, which inflates the true variance of S -
so the **uncorrected** Mann-Kendall p-value is anti-conservative. The Hamed & Rao
correction is run **beside** the plain test, and the `var_inflation` column
(corrected Var(S) / uncorrected) is the single number that says how much that
mattered. If it comes out near 1, the plain test was fine, and saying so is
itself a result.

Two things the wrapper handles that would otherwise be silent errors:

* `pymannkendall`'s Sen slope uses the array **index** as x, so on a gapped
  series it reports degC per *observation*. The `slope` column here is computed
  from the real years; `slope_index_based` is carried for comparison.
* The correction can return a non-finite Var(S) on a near-deterministic series.
  Those rows are labelled `degenerate_variance`, not left as blank cells.

SUHII is decomposed into `urban_mean` and `rural_mean` as well, because that
answers what the SUHII trend alone cannot: **did SUHII rise because the city
warmed, or because the countryside warmed less?**

In [ ]:
# COLAB: RUN THIS CELL
_suhii_csv = "data/outputs/suhii_2000_2025.csv"
if os.path.exists(_suhii_csv):
    SUHII = pd.read_csv(_suhii_csv)
    print("Loaded the Phase 3 SUHII table:", SUHII.shape)
else:
    print("Phase 3 SUHII CSV not found - rebuilding it (this costs ~42 round trips).")
    _pairs = uhi_metrics.mask_pairs(params, water=STATIC_WATER)
    SUHII = uhi_metrics.suhii_all_sources(params, pairs=_pairs, progress=True)
    os.makedirs("data/outputs", exist_ok=True)
    SUHII.to_csv(_suhii_csv, index=False)
    print("Wrote", _suhii_csv)

SUHII_TRENDS = trends.suhii_trends(SUHII, params)
os.makedirs("data/outputs", exist_ok=True)
SUHII_TRENDS.to_csv("data/outputs/suhii_trends_2000_2025.csv", index=False)
print("Wrote data/outputs/suhii_trends_2000_2025.csv")

SUHII_TRENDS[SUHII_TRENDS["series"] == "suhii"][
    ["label", "test", "n_years", "trend", "p", "slope", "var_inflation", "status"]
]

In [ ]:
# COLAB: RUN THIS CELL
_fig = viz.plot_mk_comparison(
    SUHII_TRENDS[SUHII_TRENDS["series"] == "suhii"],
    "figures/mk_comparison_suhii_2000_2025.png",
    params,
    title="SUHII trend: effect of the autocorrelation correction",
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

In [ ]:
# COLAB: RUN THIS CELL
# Per-GN series -> per-division trends, with FDR applied ACROSS divisions.
# m is ~557 here rather than ~1e5, and the tests are far less mutually
# dependent, so this is the tractable companion to the pixel-wise raster FDR.
# The spread between the two significant fractions IS the MAUP sensitivity.
_t0 = time.time()
GN_SERIES = trends.zonal_annual_series(
    TREND_SOURCE, params, level="gn", region=WORK_REGION, progress=True
)
print(f"{len(GN_SERIES)} rows in {time.time() - _t0:.0f} s")

GN_TRENDS = trends.zonal_trend_table(GN_SERIES, params)
GN_SERIES.to_csv("data/outputs/lst_by_gn_annual_2000_2025.csv", index=False)
GN_TRENDS.to_csv("data/outputs/lst_trend_by_gn_2000_2025.csv", index=False)
print("Wrote data/outputs/lst_by_gn_annual_2000_2025.csv")
print("Wrote data/outputs/lst_trend_by_gn_2000_2025.csv")
print()
_sig = int(GN_TRENDS["significant"].sum())
print(f"GN divisions with an FDR-significant trend: {_sig} of {len(GN_TRENDS)} "
      f"({_sig / max(len(GN_TRENDS), 1):.1%})")
print("Compare this with the PIXEL-wise fraction from Step 6/10. They will")
print("differ, and that spread is the aggregation-unit (MAUP) sensitivity.")
print()
GN_TRENDS.sort_values("slope", ascending=False).head(10)

## Step 10 - trend magnitude by land cover and by LCZ

**These are a present-day stratification of a historical trend.** ESA WorldCover
is 2021 and the LCZ map derives from 2018-2019 imagery, so both answer *"where is
the warming, by TODAY'S land cover?"* - not *"did land-cover change cause the
warming?"*. A pixel that was paddy in 2002 and is built now sits in the built
class for its whole history. Attribution to land-cover change is Phase 6.

In [ ]:
# COLAB: RUN THIS CELL
CLASS_TABLES = {}
for _scheme in params["trends"]["stratify"]["products"]:
    _table = trends.trend_by_class(TREND, params, WORK_REGION, _scheme)
    CLASS_TABLES[_scheme] = _table
    _csv = f"data/outputs/lst_trend_by_{_scheme}_2000_2025.csv"
    _table.to_csv(_csv, index=False)
    print("Wrote", _csv)
    display(_table)

    _fig = viz.plot_trend_by_class(
        _table, f"figures/trend_by_{_scheme}_2000_2025.png", params
    )
    print("Wrote", _fig)
    display(Image(filename=str(_fig)))

## Step 11 - RESUME: the authoritative FDR, on the exported raster

**Run this only after the Step 5 exports read COMPLETED** and the GeoTIFFs are in
`data/interim/`.

Two ways to get them there:

1. Mount Drive (`from google.colab import drive; drive.mount('/content/drive')`)
   and copy from `MyDrive/<exports.drive_folder>/`.
2. Download from Drive in a browser and upload into `data/interim/`.

This is the number that goes in the report. Step 6's preview was coarse and over
the CMC only; this is at the fit scale over the whole district.

In [ ]:
# COLAB: RUN THIS CELL  (after the exports have COMPLETED)
os.makedirs("data/interim", exist_ok=True)

_name = exports.export_name(
    "lst_trend", "district", params, res_m=FIT_SCALE, suffix=TREND_SOURCE
)
_tif = f"data/interim/{_name}.tif"

if not os.path.exists(_tif):
    raise FileNotFoundError(
        f"{_tif} not found.\n"
        "1. Check the Step 5 tasks read COMPLETED (re-run the describe_tasks cell).\n"
        f"2. Copy '{_name}.tif' from Drive folder "
        f"'{params['exports']['drive_folder']}' into data/interim/.\n"
        "   In Colab:  from google.colab import drive; drive.mount('/content/drive')\n"
        f"   then:      !cp /content/drive/MyDrive/{params['exports']['drive_folder']}/"
        f"{_name}.tif data/interim/"
    )

FDR_SUMMARY = trends.apply_fdr_to_raster(
    _tif, f"data/interim/{_name}_fdr.tif", params
)
for _k, _v in FDR_SUMMARY.items():
    print(f"  {_k:28s} {_v}")

In [ ]:
# COLAB: RUN THIS CELL
_arrays, _profile = trends.read_trend_raster(f"data/interim/{_name}_fdr.tif", params,
                                             band_order=["sen_slope", "sen_slope_fdr",
                                                         "p_two_sided", "p_adjusted",
                                                         "significant", "n_years"])
_fig = viz.plot_trend_map(
    _arrays, f"figures/trend_fdr_{TREND_SOURCE}_2000_2025.png", params,
    title=f"Sen's slope, {TREND_SOURCE}, 2000-2025 (FDR-corrected, {FIT_SCALE} m)",
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

print()
print("=" * 72)
print("HEADLINE NUMBER FOR THE REPORT")
print("=" * 72)
print(f"  {FDR_SUMMARY['fraction_of_tested']:.1%} of TESTED pixels show an "
      "FDR-significant trend")
print(f"  {FDR_SUMMARY.get('area_km2_warming', float('nan')):.1f} km2 significantly "
      "WARMING")
print(f"  {FDR_SUMMARY.get('area_km2_cooling', float('nan')):.1f} km2 significantly "
      "COOLING")
print(f"  out of {FDR_SUMMARY.get('area_km2_tested', float('nan')):.1f} km2 tested")
print()
print("Always quote the TESTED area alongside. Untested pixels are neither")
print("significant nor non-significant.")

## What to check before signing Phase 4 off

1. **Step 2a**: the probe's band names match `trends.bands` / `trends.mk_num_inputs`.
   Record them in `PROGRESS.md`.
2. **Step 2b**: `sensSlope` returned 2.0, not 0.5, for the known-slope collection.
3. **Step 2c**: record whether `ee.Reducer.kendallsCorrelation`'s own p-value is
   one- or two-sided, so `mk_p_ee` can be read correctly in future.
4. **Step 3**: the guard REJECTED the scene collection and ACCEPTED the annual
   series. If it accepted both, stop - the structural guard is broken.
5. **Step 4**: every band inside its possible range; slope percentiles do not
   saturate the palette.
6. **Step 6 vs Step 11**: preview and authoritative fractions are the same order
   of magnitude. A large gap means the preview scale is hiding structure.
7. **Step 9**: `var_inflation`. If it is far above 1, the uncorrected test was
   badly anti-conservative and only the corrected p-values may be quoted.
8. **Step 9**: the pixel-wise and per-GN significant fractions, reported as a
   pair - that spread is the MAUP sensitivity.

## What must NOT be claimed

* This is **land surface temperature**. Not air temperature, not "what residents
  feel". Surface UHI can be roughly 2x the canopy-air UHI.
* The Landsat slope is a **dry-season, ~10:30 overpass** rate. The MODIS night
  slope is a different quantity, reported beside it as a sensitivity - and it is
  weaker evidence (up to 3 K accepted uncertainty against 1 K for day).
* **C2 inter-calibration is assumed, not verified** (`landsat_c2l2.harmonisation:
  none`), and 2012-01 to 2013-03 rests on SLC-off ETM+ alone. Any cross-sensor
  trend statement carries that caveat.
* The significance map **inherits WRS-2 side-lap striping** from `obs_count`.
  Report it; do not tune it away.
* Trend stratified by land cover is **not** attribution to land-cover change.